# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/humaisali/FlyRank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> **Before running:** this notebook needs the gated warehouse release. Request access at
> [`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)
> (instant), create a **Read** token, and store it as a Colab Secret named `HF_TOKEN` — never in
> a cell. The metadata-only cells (row counts, grain checks) finish in seconds; the feature-frame
> and trap cells scan a full month across ~70 clients and may take a minute or two. If you hit
> `HTTP 429`, stop and wait — don't re-run immediately.


## 1. Unit of analysis + time window

**1. What one row means for this lane:** the raw table's grain is one row per
(`report_date`, `client_hash_id`, `content_hash_id`) — verified below in part 3a. For my lane
(Structured Content Archetype Clustering, ML-03), I collapse that up to **one row per content
item**, aggregated over a half-month window, since archetypes describe content items, not
content-days.

**2. Which table(s):** primary = `fact_content_daily_performance`, partition `month=2026-03` — a
mid-panel month, deliberately not the sealed final month (`_sample` = June 2026). Secondary =
`dim_clients`, to know which clients even have coverage in this month before trusting any
aggregate built on top of it. `dim_content`'s structural columns (content type, word count,
keyword context) are relevant to the lane but **deliberately deferred** here — I have no way to
confirm their live column names from outside the gated dataset, so extending the feature set with
them is a next step for whoever runs this, not a guess baked in now.

**3. Time window:** `report_date` between `2026-03-01` and `2026-03-31` inclusive. I split that
month at the midpoint — `2026-03-01`–`2026-03-15` ("prev") and `2026-03-16`–`2026-03-31`
("last") — so every feature window sits strictly before the window any label would be defined
on. That split matters most in part 3e.


In [ ]:
%pip -q install duckdb

import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort). Use a Colab Secret named
# HF_TOKEN so the getpass prompt never fires (a prompt left open across a Colab reconnect
# hangs the kernel).
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "fact_daily":  f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

# Metadata-only counts -- touches Parquet footers, not data, so this is cheap even though
# fact_daily is a slice of a 79M-row table.
for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:15} {n:>12,} rows")


## 2. Fields: feature / label / context / excluded

**4. Target or proxy:** my lane has no single supervised target — clustering doesn't have one
(established in ML-03). But part 3e's leakage trap needs *a* label to demonstrate the lesson on,
so I define a throwaway proxy for that demo only: `is_declining` = did a content item's
second-half-of-March impressions fall below 80% of its first-half impressions (with a minimum
first-half volume so tiny counts don't fake a "decline"). **This is not my capstone target** —
it exists purely to stage the trap honestly.

**5. One thing deliberately excluded:** the second-half-of-month aggregate
(`impressions_last`, and anything built from it) is excluded from every *honest* feature set in
this notebook, because it sits inside the very outcome window `is_declining` is defined on.
Including it anyway is exactly the mistake part 3e stages on purpose.

**Full classification:**

| Bucket | Fields | Why |
|---|---|---|
| **Feature** | `impressions_prev`, `clicks_prev`, `avg_position_prev`, `active_days_prev`, `ctr_prev` | all aggregated only from `report_date <= 2026-03-15` — knowable before the outcome window |
| **Label / proxy** | `is_declining` (demo-only) | computed from `impressions_last` vs `impressions_prev` — never a feature |
| **Context** | `content_hash_id`, `client_hash_id` | join/group keys only, never model inputs; `report_date` — windowing only |
| **Excluded** | `impressions_last`, `clicks_last`, `avg_position_last` | fall inside the outcome window — leakage risk, staged deliberately in 3e |
| **Deferred** | `dim_content` structural columns (content type, word count, keyword context) | lane-relevant, but schema not confirmed from outside the gated table yet |


In [ ]:
field_classification = {
    "feature":      ["impressions_prev", "clicks_prev", "avg_position_prev", "active_days_prev", "ctr_prev"],
    "label_proxy":  ["is_declining"],            # demo-only -- not the capstone target
    "context":      ["content_hash_id", "client_hash_id", "report_date"],
    "excluded":     ["impressions_last", "clicks_last", "avg_position_last"],
    "deferred":     ["dim_content.* (structural columns -- schema not yet confirmed)"],
}
for bucket, fields in field_classification.items():
    print(f"{bucket:12} {fields}")


## 3. Verify it with queries (grain, counts, missing values, windows)

Three verification queries (3a–3c), then the five-feature frame (3d), then the leakage trap
(3e). Every claim above gets a query below it — a contract line without a query next to it is a
guess.

### 3a. Grain — is one row really (report_date, client_hash_id, content_hash_id)?


In [ ]:
grain_violations = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {TABLES['fact_daily']}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"Rows violating the stated grain: {len(grain_violations)} (must be 0 for the grain claim to hold)")
grain_violations


### 3b. Row count and date span for `month=2026-03`


In [ ]:
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients, COUNT(DISTINCT content_hash_id) AS n_content
    FROM {TABLES['fact_daily']}
""").df()
span


### 3c. Availability — filter with `IS TRUE`, how many rows survive?

`ga4_data_available` is **three-valued**, not boolean-clean: `TRUE`, `FALSE`, or `NULL` (rows
with no access flag at all). `= FALSE` or a plain `NOT ga4_data_available` silently mishandles
the `NULL` rows, so every availability check here uses `IS TRUE` / `IS FALSE` / `IS NULL`
explicitly.


In [ ]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)  AS ga4_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS FALSE) AS ga4_unavailable_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS NULL)  AS ga4_null_flag_rows
    FROM {TABLES['fact_daily']}
""").df()

survive_pct = availability["ga4_available_rows"][0] / availability["total_rows"][0]
print(f"Rows that survive an `IS TRUE` filter: {availability['ga4_available_rows'][0]:,} ({survive_pct:.1%} of the month)")
availability


### 3d. Five-feature frame — first half of March only (`report_date <= 2026-03-15`)

Every feature is "knowable at the decision moment" for the same reason: it's aggregated
exclusively from the first half of the month, strictly before the second-half outcome window
part 3e defines a label on.

- **`impressions_prev`** — knowable because it's a plain sum of a prior-window observed signal, no future data touched.
- **`clicks_prev`** — same: summed only over the prior window.
- **`avg_position_prev`** — an average of a signal measured daily in the prior window only.
- **`active_days_prev`** — counts prior-window days with any impressions; still entirely backward-looking.
- **`ctr_prev`** — a ratio of two prior-window sums (`clicks_prev / impressions_prev`); derived, but from nothing after the decision point.


In [ ]:
features = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions)                                            AS impressions_prev,
        SUM(gsc_clicks)                                                 AS clicks_prev,
        AVG(gsc_avg_position)                                           AS avg_position_prev,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0)  AS active_days_prev,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0)         AS ctr_prev
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 50
""").df()

print(f"{len(features):,} content items with enough first-half volume to be worth featurizing")
features.head()


### 3e. The trap — add one label-derived column on purpose, watch the score jump, then remove it

Build the demo-only label from the *second* half of March, train an honest model on the 3d
features, then deliberately add `impressions_last` — literally what the label is computed
from — as a sixth feature. That is the leakage lesson from notebook 02, staged here on real
warehouse data instead of the starter CSV.


In [ ]:
outcome = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS impressions_last
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
    GROUP BY 1
""").df()

data = features.merge(outcome, on="content_hash_id", how="left")
data["impressions_last"] = data["impressions_last"].fillna(0)
data["is_declining"] = (data["impressions_last"] < 0.8 * data["impressions_prev"]).astype(int)
print("label balance (demo-only proxy, not the capstone target):")
print(data["is_declining"].value_counts(normalize=True))


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

honest_cols = ["impressions_prev", "clicks_prev", "avg_position_prev", "active_days_prev", "ctr_prev"]
model_data = data.dropna(subset=honest_cols)
X, y = model_data[honest_cols], model_data["is_declining"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler().fit(X_tr)
clf = LogisticRegression(max_iter=1000).fit(scaler.transform(X_tr), y_tr)
honest_auc = roc_auc_score(y_te, clf.predict_proba(scaler.transform(X_te))[:, 1])
print(f"HONEST features only (5 features from 3d) -> ROC-AUC: {honest_auc:.3f}")


In [ ]:
# THE TRAP: add impressions_last -- literally what is_declining is computed FROM.
leaky_cols = honest_cols + ["impressions_last"]
model_data2 = data.dropna(subset=leaky_cols)
X2, y2 = model_data2[leaky_cols], model_data2["is_declining"]
X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y2, test_size=0.3, random_state=42, stratify=y2)

scaler2 = StandardScaler().fit(X2_tr)
clf2 = LogisticRegression(max_iter=1000).fit(scaler2.transform(X2_tr), y2_tr)
leaky_auc = roc_auc_score(y2_te, clf2.predict_proba(scaler2.transform(X2_te))[:, 1])
print(f"LEAKY (+impressions_last)          -> ROC-AUC: {leaky_auc:.3f}")
print(f"Jump from one label-derived column: {leaky_auc - honest_auc:+.3f}")


In [ ]:
# Remove the leak and keep the honest number -- this is the number that goes in the report.
del leaky_cols, X2, y2, X2_tr, X2_te, y2_tr, y2_te, clf2, leaky_auc

print(f"KEPT: honest ROC-AUC with only prior-window features = {honest_auc:.3f}")
print("The leaky run is deleted above, on purpose -- it never gets reported as a real result.")


## 4. Data limits

**Named limitation: GA4 coverage is genuinely partial within this month, not just missing at
random.** Part 3c's `IS TRUE` / `IS FALSE` / `IS NULL` split isn't a formality — some clients
have no GA4 tracking at all yet (`ga4_data_start` is `NULL` in `dim_clients`), so entire rows for
those clients carry a `NULL` availability flag all month, every month. Any engagement-flavored
archetype (e.g. "engagement-problem pages") built from this slice will systematically under-cover
those clients — not because their pages don't have engagement problems, but because the
instrumentation to see it was never turned on. That's a fact about the panel, not about the
pages, and it has to be named before any cluster gets interpreted as "these clients don't
engage."


In [ ]:
client_coverage = con.sql(f"""
    SELECT
        COUNT(*)                                   AS n_clients,
        COUNT(*) FILTER (WHERE ga4_data_start IS NULL)                        AS n_no_ga4_ever,
        COUNT(*) FILTER (WHERE ga4_data_start > DATE '2026-03-01')          AS n_ga4_starts_after_march
    FROM {TABLES['dim_clients']}
""").df()

print(f"Of {client_coverage['n_clients'][0]} clients: "
      f"{client_coverage['n_no_ga4_ever'][0]} have no GA4 tracking at all, and "
      f"{client_coverage['n_ga4_starts_after_march'][0]} more only start GA4 after this month begins.")
client_coverage


## Self-check

Before you submit, confirm each line honestly. **These are unchecked on purpose** — I (the AI
assistant) drafted this notebook but could not execute it: my sandbox can't reach
`huggingface.co`, and I shouldn't be handling your `HF_TOKEN` either way. You need to actually
run this in Colab and check these yourself:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
